# Walkthrough: synthetic data to a scored forecast

This notebook runs the same steps as `python -m src.train`, one at a time, on a small **synthetic** dataset generated in the repo. Nothing here comes from a real retailer, and the numbers it prints say nothing about accuracy on real data.

Run it from the repository root after `pip install -e .` (for example `jupyter lab` from the root, or add the root to `sys.path`).

In [ ]:
from datetime import timedelta

import polars as pl

from src.data.loader import merge_m5_frames
from src.data.synthetic import SyntheticDataConfig, SyntheticDataGenerator
from src.data.validators import DataValidator

sales, calendar, prices = SyntheticDataGenerator(
    SyntheticDataConfig(n_items=21, n_stores=5, n_days=730, random_seed=42)
).generate_all()
data = merge_m5_frames(sales, calendar, prices)
print(data.shape)
data.head()

## 1. Validate before anything else

Pandera schemas check types and ranges; plain rules check duplicates and history length. Nulls in optional columns such as event names are warnings, not errors.

In [ ]:
result = DataValidator(min_history_days=84).validate(data, "sales")
print("valid:", result.is_valid)
print("warnings:", result.warnings[:3])

## 2. Hide the future, then build features

The forecast origin is 28 days before the end. Target values after it are set to null *before* any feature is built, and every target-based feature is at least `min_lag = 28` days old. That is what makes the evaluation honest.

In [ ]:
from src.features.engineer import FeatureEngineer, FeatureEngineerConfig, to_model_input

origin = data["date"].max() - timedelta(days=28)
masked = data.with_columns(
    pl.when(pl.col("date") > origin).then(None).otherwise(pl.col("sales")).alias("sales")
)
engineer = FeatureEngineer(FeatureEngineerConfig(min_lag=28))
features = engineer.fit_transform(masked)
columns = engineer.get_feature_names()
engineer.get_feature_summary()["group_counts"]

## 3. Train LightGBM with early stopping on the last 28 days before the origin

In [ ]:
from src.models.lightgbm_model import LightGBMConfig, LightGBMForecaster

val_start = origin - timedelta(days=27)
fit_part = features.filter(pl.col("date") < val_start)
val_part = features.filter(pl.col("date").is_between(val_start, origin))
test_part = features.filter(pl.col("date") > origin)

model = LightGBMForecaster(config=LightGBMConfig(n_estimators=400, num_leaves=31, min_data_in_leaf=50))
model.fit(
    to_model_input(fit_part, columns), fit_part["sales"].to_numpy(),
    X_valid=to_model_input(val_part, columns), y_valid=val_part["sales"].to_numpy(),
)
model.get_feature_importance().head(10)

## 4. Score the forecast with WRMSSE and compare with naive baselines

WRMSSE is the M5 competition metric: errors are scaled by each series' own day-to-day volatility, weighted by recent dollar sales, and averaged over 12 levels of the hierarchy. Lower is better; 0 is perfect.

In [ ]:
import pandas as pd

from src.train import moving_average, score, seasonal_naive

predictions = model.predict(to_model_input(test_part, columns))
train_pd = data.filter(pl.col("date") <= origin).drop("d").to_pandas()
test_pd = data.filter(pl.col("date") > origin).sort(["id", "date"]).drop("d").to_pandas()
for frame in (train_pd, test_pd):
    frame["date"] = pd.to_datetime(frame["date"])

rows = {
    "model": score(train_pd, test_pd, predictions),
    "seasonal_naive_7d": score(train_pd, test_pd, seasonal_naive(train_pd, test_pd)),
    "moving_average_28d": score(train_pd, test_pd, moving_average(train_pd, test_pd)),
}
pd.DataFrame(rows).T[["wrmsse", "wrmsse_series_level", "rmse", "mae"]]

## What this does not show

- Accuracy on real data. Run `python -m src.train --data-dir data/raw --stores CA_1` on the M5 files for that.
- Uncertainty. `predict_interval` is a fixed ±20% heuristic, not a calibrated interval.
- Anything about serving: the API does not yet look up the recent sales history the model needs.